# J-space language autoencoder — Colab runner (L4)

Tests one cycle end to end:

```
J-space cone q -> cone adapter -> frozen Gemma writes a phrase
                                        |
      q_hat <- frozen phrase reconstructor
```

Only the **cone adapter** and the **phrase reconstructor** are trained. Gemma,
the tokenizer, the fitted lens, the J-space dictionary, and the k=10 pursuit are
frozen and asserted so at every stage.

Sections:

1. Setup and checksums (runtime, repo, HF auth, Drive, lens verification, tests)
2. Dataset smoke build
3. Reconstructor training
4. Reconstructor gate
5. Adapter warm start
6. Reconstructor-guided preference training
7. Held-out evaluation
8. Artifact export
9. Final GO/NO-GO report

Every long stage runs as a background process teeing to a Drive-backed log, so
losing the browser tab does not lose the run, and every stage resumes from its
own checkpoints.

> **This notebook does not run the full pilot automatically.** Sections 2-7
> default to `--smoke` (the deterministic CPU mock, no checkpoint, no
> downloads). The real L4 pilot commands are printed by section 1 and are
> enabled by setting `SMOKE = False` in the configuration cell.

## 1. Setup and checksums

No model load in this section. It fails loudly rather than proceeding on an
unverified lens or a stale checkout.

In [ ]:
# 1a. Runtime facts: Python, PyTorch, Transformers, CUDA, GPU, memory.
import platform
import shutil
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
print(f"IN_COLAB          = {IN_COLAB}")
print(f"Python            = {platform.python_version()}  ({sys.executable})")

try:
    import torch
except ImportError:
    torch = None
    print("PyTorch           = NOT INSTALLED")
else:
    print(f"PyTorch           = {torch.__version__}")
    print(f"torch.version.cuda= {torch.version.cuda}")
    print(f"cuda_available    = {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        properties = torch.cuda.get_device_properties(0)
        print(f"GPU               = {properties.name}")
        print(f"GPU memory        = {properties.total_memory / 2**30:.1f} GiB")

try:
    import transformers
except ImportError:
    print("transformers      = NOT INSTALLED")
else:
    print(f"transformers      = {transformers.__version__}")

if shutil.which("nvidia-smi"):
    print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
else:
    print("nvidia-smi unavailable")

In [ ]:
# 1b. Clone or update the branch, assert HEAD, install the package.
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/MechInterpreter/jacobian-lens-gemma.git"
BRANCH = "experiment/jspace-language-autoencoder"
CHECKOUT_DIR = Path("/content/jacobian-lens-gemma") if IN_COLAB else Path.cwd()
EXPECTED_HEAD = None  # set to a full 40-char sha to pin an exact commit


def run(argv, cwd=None, check=True):
    result = subprocess.run(
        argv, cwd=str(cwd) if cwd else None, capture_output=True, text=True
    )
    if check and result.returncode != 0:
        raise RuntimeError(f"{' '.join(argv)} failed:\n{result.stdout}\n{result.stderr}")
    return result.stdout.strip()


if IN_COLAB:
    if not (CHECKOUT_DIR / ".git").is_dir():
        run(["git", "clone", "--branch", BRANCH, REPO_URL, str(CHECKOUT_DIR)])
    else:
        run(["git", "fetch", "origin", BRANCH], cwd=CHECKOUT_DIR)
        run(["git", "checkout", BRANCH], cwd=CHECKOUT_DIR)
        run(["git", "reset", "--hard", f"origin/{BRANCH}"], cwd=CHECKOUT_DIR)

branch = run(["git", "rev-parse", "--abbrev-ref", "HEAD"], cwd=CHECKOUT_DIR)
head = run(["git", "rev-parse", "HEAD"], cwd=CHECKOUT_DIR)
print(f"branch = {branch}")
print(f"HEAD   = {head}")
if branch != BRANCH:
    raise RuntimeError(f"HEAD assertion failed: on {branch!r}, expected {BRANCH!r}")
if EXPECTED_HEAD and head != EXPECTED_HEAD:
    raise RuntimeError(f"HEAD assertion failed: {head} != pinned {EXPECTED_HEAD}")

run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], cwd=CHECKOUT_DIR)
if str(CHECKOUT_DIR) not in sys.path:
    sys.path.insert(0, str(CHECKOUT_DIR))
os.chdir(CHECKOUT_DIR)
print("installed and on sys.path")

In [ ]:
# 1c. Semantic HEAD assertion: the autoencoder API this notebook drives must exist.
import importlib

import jlens.autoencoder as jae
import jlens.autoencoder.evaluation as jeval

importlib.reload(jae)
REQUIRED_API = [
    (jae, "AutoencoderConfig"),
    (jae, "load_autoencoder_config"),
    (jeval, "gonogo_report"),
    (jeval, "attribute_failure"),
]
missing = [f"{m.__name__}.{n}" for m, n in REQUIRED_API if not hasattr(m, n)]
if missing:
    raise RuntimeError(
        f"this checkout predates the J-space language autoencoder API: missing {missing}"
    )
print("autoencoder API present")

In [ ]:
# 1d. Hugging Face token via getpass. Never printed, never persisted.
import getpass
import os

if os.environ.get("HF_TOKEN"):
    print("HF_TOKEN already set for this process - leaving it alone.")
else:
    _token = getpass.getpass("Hugging Face token (input hidden, blank to skip): ").strip()
    if _token:
        os.environ["HF_TOKEN"] = _token
        del _token
    else:
        print("No token entered. Smoke mode works without one; the real pilot does not.")

_value = os.environ.get("HF_TOKEN", "")
print(f"HF_TOKEN set = {bool(_value)}  (length {len(_value)}, prefix {_value[:3]}***)")
del _value

In [ ]:
# 1e. Mount Drive and resolve the persistent paths. Creates, never deletes.
from pathlib import Path

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive", force_remount=False)
    DRIVE_MOUNT = Path("/content/drive")
    if not (DRIVE_MOUNT / "MyDrive").is_dir():
        raise RuntimeError("Drive did not mount: /content/drive/MyDrive is missing.")
    DRIVE_ROOT = DRIVE_MOUNT / "MyDrive" / "jacobian-lens-gemma"
else:
    DRIVE_ROOT = Path.cwd() / "local_drive"

RUNS_ROOT = DRIVE_ROOT / "runs"          # holds the pilot run with lens.pt
JLANG_ROOT = DRIVE_ROOT / "jlang_runs"   # this experiment's run directories
LOG_ROOT = DRIVE_ROOT / "logs"
ARCHIVE_ROOT = DRIVE_ROOT / "archives"
for path in (RUNS_ROOT, JLANG_ROOT, LOG_ROOT, ARCHIVE_ROOT):
    path.mkdir(parents=True, exist_ok=True)
    print(f"{path}  (exists={path.is_dir()})")

In [ ]:
# 1f. Experiment configuration for this notebook session.
#
# SMOKE = True  -> deterministic CPU mock: no checkpoint, no downloads, minutes.
# SMOKE = False -> the real L4 pilot: gated checkpoint, verified lens, hours.
SMOKE = True

CONFIG_PATH = "configs/jspace_language_autoencoder.yaml"
RUN_NAME = "jlang_smoke" if SMOKE else "jlang_pilot"
RUN_DIR = JLANG_ROOT / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)

COMMON_ARGS = ["--config", CONFIG_PATH, "--output-dir", str(RUN_DIR)]
if SMOKE:
    COMMON_ARGS += ["--smoke"]
else:
    COMMON_ARGS += [
        "--allow-model-load",
        "--device-map", "cuda",
        "--runs-root", str(RUNS_ROOT),
    ]

print(f"SMOKE   = {SMOKE}")
print(f"RUN_DIR = {RUN_DIR}")
print("\nExact pilot commands (SMOKE = False):\n")
for script, extra in [
    ("build_jspace_language_dataset", ["--benchmark"]),
    ("train_phrase_reconstructor", []),
    ("train_cone_adapter", []),
    ("evaluate_jspace_language", []),
]:
    argv = [
        "python", "-u", f"scripts/{script}.py",
        "--config", CONFIG_PATH,
        "--output-dir", str(JLANG_ROOT / "jlang_pilot"),
        "--allow-model-load", "--device-map", "cuda",
        "--runs-root", str(RUNS_ROOT),
        *extra,
    ]
    print("  " + " ".join(argv) + "\n")

In [ ]:
# 1g. Lens verification. The pilot lens is a hard gate for the real run and is
# not needed in smoke mode (the mock builds its own deterministic lens).
import hashlib

from jlens.autoencoder.config import load_autoencoder_config

CONFIG = load_autoencoder_config(CONFIG_PATH)
LENS_PATH = RUNS_ROOT / CONFIG.lens.run_dir_name / CONFIG.lens.artifact_relpath
EXPECTED_LENS_SHA256 = CONFIG.lens.expect_file_sha256

if SMOKE:
    print("smoke mode: the mock stack builds its own lens; skipping verification.")
    LENS_OK = True
else:
    if not LENS_PATH.is_file():
        raise RuntimeError(
            f"ABORT - lens not found at {LENS_PATH}. Upload the pilot run's "
            f"artifacts/lens.pt there before running the pilot."
        )
    digest = hashlib.sha256()
    with open(LENS_PATH, "rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b""):
            digest.update(chunk)
    observed = "sha256:" + digest.hexdigest()
    LENS_OK = observed == EXPECTED_LENS_SHA256
    print(f"lens   = {LENS_PATH}")
    print(f"bytes  = {LENS_PATH.stat().st_size:,}")
    print(f"sha256 = {observed}")
    if not LENS_OK:
        raise RuntimeError(
            f"ABORT - lens checksum mismatch: expected {EXPECTED_LENS_SHA256}"
        )
    print("lens verified.")

print(f"config fingerprint = {CONFIG.fingerprint()}")

In [ ]:
# 1h. Run the autoencoder test suite on the CPU mock. This is the same code the
# pilot runs, minus the checkpoint; a failure here invalidates everything below.
import subprocess
import sys

result = subprocess.run(
    [sys.executable, "-m", "pytest", "-q",
     "tests/test_jspace_language_config.py",
     "tests/test_jspace_language_dataset.py",
     "tests/test_jspace_language_models.py",
     "tests/test_jspace_language_eval.py",
     "tests/test_jspace_language_e2e.py"],
    capture_output=True, text=True,
)
print(result.stdout[-4000:])
print(result.stderr[-2000:])
if result.returncode != 0:
    raise RuntimeError("ABORT - the autoencoder test suite failed on this checkout.")

In [ ]:
# 1i. Streaming stage runner: background process + Drive-backed log + resume.
# Interrupting a monitoring cell does NOT stop a stage; rerun the monitor cell
# to reattach, or rerun the stage cell to resume from its checkpoints.
import datetime
import subprocess
import sys
import threading
import time
from pathlib import Path

STAGES = {}


def launch(stage: str, script: str, extra_args=()):
    """Start a stage in the background, teeing stdout+stderr to a Drive log."""
    stamp = datetime.datetime.now().strftime("%Y%m%dT%H%M%S")
    log_path = LOG_ROOT / f"{RUN_NAME}_{stage}_{stamp}.log"
    argv = [sys.executable, "-u", f"scripts/{script}.py", *COMMON_ARGS, *extra_args]
    print(" ".join(argv))
    handle = open(log_path, "w", encoding="utf-8", buffering=1)
    process = subprocess.Popen(
        argv, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
    )

    def pump():
        for line in process.stdout:
            handle.write(line)
        handle.flush()
        handle.close()

    thread = threading.Thread(target=pump, daemon=True)
    thread.start()
    STAGES[stage] = {"process": process, "log_path": log_path, "thread": thread}
    print(f"launched {stage}, log -> {log_path}")
    return STAGES[stage]


def follow(stage: str, poll_seconds: float = 2.0, gpu_every: int = 10):
    """Stream a running stage to completion, with periodic GPU state."""
    entry = STAGES.get(stage)
    if entry is None:
        raise RuntimeError(f"stage {stage!r} was never launched")
    log_path = Path(entry["log_path"])
    position = 0
    ticks = 0
    started = time.time()
    while True:
        if log_path.exists():
            with open(log_path, encoding="utf-8") as handle:
                handle.seek(position)
                chunk = handle.read()
                position = handle.tell()
            if chunk:
                print(chunk, end="")
        finished = entry["process"].poll() is not None
        ticks += 1
        if gpu_every and ticks % gpu_every == 0:
            print(f"[monitor] {time.time() - started:.0f}s elapsed | {gpu_line()}")
        if finished:
            break
        time.sleep(poll_seconds)
    entry["thread"].join(timeout=10)
    code_ = entry["process"].returncode
    print(f"\n[{stage}] exit code {code_} after {time.time() - started:.0f}s")
    return code_


def gpu_line():
    import shutil

    if not shutil.which("nvidia-smi"):
        return "nvidia-smi unavailable"
    query = subprocess.run(
        ["nvidia-smi",
         "--query-gpu=memory.used,memory.total,utilization.gpu,temperature.gpu",
         "--format=csv,noheader,nounits"],
        capture_output=True, text=True,
    )
    return "GPU " + query.stdout.strip().replace("\n", " | ")


print("runner ready")

## 2. Dataset smoke build

Mines 2-6 Gemma-token phrases, collects several natural occurrences of each,
captures the layer-14 activation immediately before each occurrence, and runs
the frozen k=10 pursuit.

`--benchmark` measures one small slice first and prints **measured** seconds per
occurrence plus a linearly extrapolated wall time and storage estimate, before
the full build is attempted.

Splits are by phrase identity: every occurrence of a phrase lands in the same
split, and `artifacts/leakage.json` re-derives the assignment and fails on any
crossing.

In [ ]:
# 2. Build the dataset (resumable by rerunning: the cache is written atomically).
launch("dataset", "build_jspace_language_dataset", ["--benchmark"])
DATASET_CODE = follow("dataset")
if DATASET_CODE != 0:
    raise RuntimeError(f"dataset build failed (exit {DATASET_CODE}); see the log above")

In [ ]:
# 2b. What was built: split sizes, phrase counts, and the measured cost.
import json

manifest = json.loads((RUN_DIR / "dataset" / "manifest.json").read_text(encoding="utf-8"))
leakage = json.loads((RUN_DIR / "artifacts" / "leakage.json").read_text(encoding="utf-8"))
print(f"records = {manifest['n_records']}, phrases = {manifest['n_phrases']}")
print(f"phrase splits = {manifest['stats']['phrase_split_counts']}")
print(f"leakage clean = {leakage['clean']}  ({len(leakage['violations'])} violations)")

benchmark_path = RUN_DIR / "artifacts" / "benchmark.json"
if benchmark_path.exists():
    benchmark = json.loads(benchmark_path.read_text(encoding="utf-8"))
    measured = benchmark["measured"]
    projection = benchmark["projection"]
    print(f"\nmeasured: {measured['seconds_per_occurrence']:.4f} s/occurrence, "
          f"peak CUDA {measured['peak_cuda_memory_gb']} GB")
    print(f"projected full build: {projection['planned_occurrences']} occurrences, "
          f"{projection['estimated_wall_minutes']:.1f} min, "
          f"{projection['estimated_storage_mb']:.1f} MB "
          f"({projection['basis']})")

## 3. Reconstructor training

Trains the phrase reconstructor on **training-split prototypes only**, then
freezes it permanently. It is never jointly adapted with the cone adapter.

In [ ]:
# 3. Train the reconstructor. --ignore-gate records the override in the
# artifacts; remove it to make a failed gate stop the notebook here.
launch("reconstructor", "train_phrase_reconstructor", ["--ignore-gate"])
RECONSTRUCTOR_CODE = follow("reconstructor")
print(f"exit {RECONSTRUCTOR_CODE} (0 = trained; 3 = trained but the gate failed)")

## 4. Reconstructor gate

**The stop condition.** If the reconstructor cannot separate correct phrases
from hard distractors on concept-disjoint validation data, the verbalizer must
not be trained: nothing downstream would be interpretable.

- correct-vs-distractor AUROC >= 0.80
- correct-phrase top-5 retrieval >= 50%

In [ ]:
# 4. The gate verdict, with each criterion's observed value next to its threshold.
import json

gate = json.loads((RUN_DIR / "artifacts" / "reconstructor_gate.json").read_text(encoding="utf-8"))
metrics = json.loads(
    (RUN_DIR / "artifacts" / "reconstructor_metrics.json").read_text(encoding="utf-8")
)

print(f"VERDICT: {gate['verdict']}")
print(gate["message"])
print()
for criterion in gate["criteria"]:
    observed = criterion["observed"]
    shown = "n/a" if observed is None else f"{observed:.4f}"
    print(f"  {criterion['name']:<32} {shown:>8}  >= {criterion['threshold']:<6} "
          f"{'PASS' if criterion['passed'] else 'FAIL'}")

print()
header = f"{'split':<9}{'auroc':>8}{'top1':>8}{'top5':>8}{'explained':>11}{'margin':>9}"
print(header)
for split, values in metrics.items():
    auroc = values["auroc_correct_vs_distractor"]
    print(f"{split:<9}{('n/a' if auroc is None else f'{auroc:.3f}'):>8}"
          f"{values['top1_retrieval']:>8.3f}{values['top5_retrieval']:>8.3f}"
          f"{values['mean_explained_fraction']:>11.4f}"
          f"{values['mean_specificity_margin']:>9.4f}")

GATE_PASSED = bool(gate["passed"])
if not GATE_PASSED:
    print("\nNOTE: the gate did NOT pass. Continuing past this point produces a "
          "recorded NO-GO run, not a result about verbalization.")

## 5. Adapter warm start

Supervised teacher-forced phrase cross-entropy through frozen Gemma. The
end-of-turn token is part of the target, so the adapter learns to stop.

Gemma stays frozen: the optimizer's parameter identities are checked against the
frozen modules before the first step, and Gemma is re-checked for gradients
after every backward pass.

Every epoch writes `adapter_epoch<NN>.pt` with optimizer and RNG state, so a
terminated runtime costs at most one epoch — rerun the cell to resume.

In [ ]:
# 5. Warm start only (preference training is section 6).
launch("adapter_warm", "train_cone_adapter", ["--skip-preference"])
WARM_CODE = follow("adapter_warm")
if WARM_CODE != 0:
    raise RuntimeError(f"adapter warm start failed (exit {WARM_CODE})")

## 6. Reconstructor-guided preference training

The reconstructor now *optimizes* the verbalizer rather than filtering it after
the fact: beam candidates are mapped back to J-space, scored against `q` and
against unrelated cones, and the adapter is updated with an offline pairwise
preference loss anchored to a frozen copy of the warm-start adapter.

Nothing backpropagates into Gemma or the reconstructor. Policy-gradient
refinement exists and is disabled by default.

In [ ]:
# 6. Full adapter training: warm start (resumed/skipped if complete) + preference.
launch("adapter", "train_cone_adapter", [])
ADAPTER_CODE = follow("adapter")
if ADAPTER_CODE != 0:
    raise RuntimeError(f"adapter training failed (exit {ADAPTER_CODE})")

In [ ]:
# 6b. Training curves: warm-start NLL and preference loss / reward / abstention.
import json

training = json.loads(
    (RUN_DIR / "artifacts" / "adapter_training.json").read_text(encoding="utf-8")
)
print("warm start")
for entry in training["warm_start"].get("history", []):
    print(f"  epoch {entry['epoch']:>3}  loss {entry['loss']:.4f}  "
          f"target tokens {entry['n_target_tokens']}")
print("preference")
for entry in training["preference"].get("history", []):
    print(f"  epoch {entry['epoch']:>3}  loss {entry['loss']:.4f}  "
          f"pairs {entry['n_pairs']}  best reward {entry['mean_best_reward']:.4f}  "
          f"abstention {entry['abstention_rate']:.2f}")

## 7. Held-out evaluation

Concept-disjoint held-out phrases only. Runs all eight baselines through the
identical prompt, beam width, and scorer — only the memory differs — plus the
paraphrase sweep, the cross-cone swap check, and the confabulation-attractor
probe (`black hole`, `photosynthesis`, `quantum entanglement`,
`Great Barrier Reef`).

Exit code 0 = GO, 4 = NO-GO. Both write the full artifacts.

In [ ]:
# 7. Held-out evaluation.
launch("evaluation", "evaluate_jspace_language", [])
EVAL_CODE = follow("evaluation")
print(f"exit {EVAL_CODE} (0 = GO, 4 = NO-GO, 2 = aborted)")
if EVAL_CODE not in (0, 4):
    raise RuntimeError(f"evaluation aborted (exit {EVAL_CODE}); see the log above")

In [ ]:
# 7b. What the model actually said, per held-out record, per baseline.
import json

evaluation = json.loads(
    (RUN_DIR / "artifacts" / "evaluation.json").read_text(encoding="utf-8")
)


def table(rows, headers):
    if not rows:
        print("(no rows)")
        return
    cells = [[("-" if v is None else str(v)) for v in row] for row in rows]
    widths = [max(len(headers[i]), max(len(r[i]) for r in cells)) for i in range(len(headers))]
    print("  ".join(h.ljust(w) for h, w in zip(headers, widths)))
    print("  ".join("-" * w for w in widths))
    for row in cells:
        print("  ".join(c.ljust(w) for c, w in zip(row, widths)))


rows = []
for record in evaluation["per_record"]:
    reranked = record["results"].get("adapter_reranked", {})
    zero = record["results"].get("zero_memory", {})
    rows.append([
        record["phrase"],
        reranked.get("top_text"),
        None if reranked.get("top_cosine") is None else f"{reranked['top_cosine']:.3f}",
        reranked.get("verdict"),
        zero.get("top_text"),
        (record.get("first_token") or {}).get("first_token_rank"),
    ])
table(rows, ["phrase", "adapter says", "cos", "verdict", "zero-memory says", "1st tok rank"])

print()
print(f"cross-cone swap: {evaluation['cross_cone_swap']}")
print(f"resources: {evaluation['resources']}")

## 8. Artifact export

Zips the run directory and its logs into `MyDrive/jacobian-lens-gemma/archives/`
and fingerprints the archive. Previous runs and archives are never deleted.

In [ ]:
# 8. Archive the run + logs to Drive and fingerprint the archive.
import hashlib
import zipfile

ARCHIVE_PATH = ARCHIVE_ROOT / f"{RUN_NAME}.zip"
n_files = 0
with zipfile.ZipFile(ARCHIVE_PATH, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(RUN_DIR.rglob("*")):
        if path.is_file():
            archive.write(path, arcname=str(Path(RUN_NAME) / path.relative_to(RUN_DIR)))
            n_files += 1
    for entry in STAGES.values():
        log_path = Path(entry["log_path"])
        if log_path.is_file():
            archive.write(log_path, arcname=str(Path(RUN_NAME) / "logs" / log_path.name))
            n_files += 1

digest = hashlib.sha256()
with open(ARCHIVE_PATH, "rb") as handle:
    for chunk in iter(lambda: handle.read(1 << 20), b""):
        digest.update(chunk)
print(f"archive = {ARCHIVE_PATH}")
print(f"files   = {n_files}")
print(f"bytes   = {ARCHIVE_PATH.stat().st_size:,}")
print(f"sha256  = {digest.hexdigest()}")

## 9. Final GO/NO-GO report

All seven criteria on concept-disjoint held-out phrases, each with its observed
value next to its threshold. A NO-GO is attributed to exactly one primary
failure mode with the evidence for that attribution.

There is no code path that hides a negative result.

In [ ]:
# 9. The verdict.
import json

report = json.loads((RUN_DIR / "artifacts" / "gonogo.json").read_text(encoding="utf-8"))
print((RUN_DIR / "summary.md").read_text(encoding="utf-8"))

print("\nartifact identity")
for key, value in report["artifact_identity"].items():
    print(f"  {key:<24} {value}")

if report.get("confabulation_probe"):
    print("\nconfabulation attractors (must not score highly on unrelated cones)")
    for entry in report["confabulation_probe"]["per_attractor"]:
        print(f"  {entry['phrase']:<24} mean {entry['mean_cosine_vs_unrelated_cones']:+.3f}  "
              f"above threshold {entry['fraction_above_accept_threshold']:.2%}")
    print(f"  probe clean = {report['confabulation_probe']['clean']}")

if report.get("prompt_robustness"):
    print("\nprompt-paraphrase robustness")
    for prompt_id, metrics in report["prompt_robustness"]["per_prompt"].items():
        print(f"  {prompt_id:<28} top-1 {metrics['exact_match_top1']:.3f}")
    print(f"  cross-prompt agreement = {report['prompt_robustness']['cross_prompt_agreement']}")

if not report["passed"]:
    attribution = report["failure_attribution"]
    print(f"\nPRIMARY FAILURE MODE: {attribution['primary']}")
    print(attribution["reason"])
    print("\nevidence:")
    for key, value in attribution["evidence"].items():
        print(f"  {key:<36} {value}")